## This notebook deals with working with UDF (User Defined Functions) for custom requirements

In [0]:
data = [
    ("Order ID: 56789",),
    ("Invoice# 98321 processed",),
    ("No numbers here",),
    ("Ref-45A12B",)
]

df = spark.createDataFrame(data, ["text_col"])
df.createOrReplaceTempView("orders")
display(df)

### Why Scala/Java UDFs are generally faster than Python UDFs

- Spark runs on the **JVM**, so Scala/Java UDFs execute **inside the executor JVM** itself.
- Scala/Java UDFs are compiled to **JVM bytecode** and run natively on the worker nodes, with no extra process or data movement.
- The **driver** sends the job (including the UDF) to the **executors** on the worker nodes.
- **Python UDFs** can't run inside the JVM. Each executor has to start a separate **Python worker process** to run the code.
- For every row (or batch), data is **serialized** from the JVM, sent to the Python process, processed, and then **serialized back** to the JVM.
- This serialization and inter-process communication adds **significant overhead**, which makes Python UDFs slower.
- **Spark's Catalyst optimizer** also can't see inside a UDF, so it can't optimize it, whether the UDF is written in Python or Scala.

#### Ways to reduce the cost in Python
- Prefer **built-in Spark SQL / DataFrame functions** whenever possible.
- Use **Pandas UDFs (vectorized UDFs)**, which use Apache Arrow to transfer data in batches and are much faster than row-at-a-time Python UDFs.

In [0]:
import re

#define the python function for extraction
def extract_number(text):
    if text is None:
        return None
    match=re.search(r'\d+', text) # returns first sequence of digits
    if match:
        return match.group()
    else:
        return None
    
#register the function as UDF
spark.udf.register("extract_number",extract_number)

This scala code will not work since we are running it on Serverless compute cluster 

In [0]:
import org.apache.spark.sql.functions.udf

// define the function for extraction
val extractNumber = (text: String) => {
  if (text == null) None
  else "\\d+".r.findFirstIn(text)   // returns Option[String], first sequence of digits
}

// register the function as UDF so it can be used in SQL
spark.udf.register("extract_number", extractNumber)

// optional: also use it directly in the DataFrame API
val extractNumberUdf = udf(extractNumber)

In [0]:
from pyspark.sql.functions import expr

df_udf=df.withColumn("extracted_number",expr("extract_number(text_col)"))
display(df_udf)

In [0]:
%sql
select 
    *, extract_number(text_col) as extracted_number
from orders